# Modelo com implementação própria

## Bibliotecas

Não foram utilizadas bibliotecas de machine learning ou deep learning. O modelo foi implementado utilizando NumPy para os cálculos numéricos e Pandas para a leitura do dataset e criação do ficheiro de submissão. Foram também utilizadas as bibliotecas copy e re, nas funções implementadas nas aulas.

In [1]:
import pandas as pd
import numpy as np
import copy
import re

from collections import Counter
from abc import ABCMeta, abstractmethod

## Funções das aulas adaptadas

O modelo foi implementado com base em código adaptado a partir dos exemplos fornecidos nas aulas (2) e (3), tendo sido realizadas as devidas modificações para se adequar aos requesitos específicos do problema.

In [2]:
class Data:
    def __init__(self, X, y = None, features = None, label= None):
        if X is None:
            raise ValueError("X cannot be None")
        if y is not None and len(X) != len(y):
            raise ValueError("X and y must have the same length")
        if features is not None and len(X[0]) != len(features):
            raise ValueError("Number of features must match the number of columns in X")
        if features is None:
            features = [f"feat_{str(i)}" for i in range(X.shape[1])]
        if y is not None and label is None:
            label = "y"
        self.X = X
        self.y = y
        self.features = features
        self.label = label

    def shape(self):
        return self.X.shape

    def has_label(self):
        return self.y is not None

    def get_classes(self):
        if self.has_label():
            return np.unique(self.y)
        else:
            raise ValueError("Dataset does not have a label")

    def get_mean(self):
        return np.nanmean(self.X, axis=0)

    def get_variance(self):
        return np.nanvar(self.X, axis=0)

    def get_median(self):
        return np.nanmedian(self.X, axis=0)

    def get_min(self):
        return np.nanmin(self.X, axis=0)

    def get_max(self):
        return np.nanmax(self.X, axis=0)

    def summary(self):
        data = {
            "mean": self.get_mean(),
            "median": self.get_median(),
            "min": self.get_min(),
            "max": self.get_max(),
            "var": self.get_variance()
        }
        return pd.DataFrame.from_dict(data, orient="index", columns=self.features)

In [3]:
class Layer(metaclass=ABCMeta):

    @abstractmethod
    def forward_propagation(self, input):
        raise NotImplementedError
    
    @abstractmethod
    def backward_propagation(self, error):
        raise NotImplementedError
    
    @abstractmethod
    def output_shape(self):
        raise NotImplementedError
    
    @abstractmethod
    def parameters(self):
        raise NotImplementedError
    
    def set_input_shape(self, input_shape):
        self._input_shape = input_shape

    def input_shape(self):
        return self._input_shape
    
    def layer_name(self):
        return self.__class__.__name__

In [4]:
class DenseLayer (Layer):
    
    def __init__(self, n_units, input_shape = None, l2_lambda = 0.01):
        super().__init__()
        self.n_units = n_units
        self._input_shape = input_shape
        self.l2_lambda = l2_lambda

        self.input = None
        self.output = None
        self.weights = None
        self.biases = None
        
    def initialize(self, optimizer):
        self.weights = np.random.randn(self.input_shape()[0], self.n_units) * np.sqrt(2./self.input_shape()[0])
        self.biases = np.zeros((1, self.n_units))
        self.w_opt = copy.deepcopy(optimizer)
        self.b_opt = copy.deepcopy(optimizer)
        return self
    
    def parameters(self):
        return np.prod(self.weights.shape) + np.prod(self.biases.shape)

    def forward_propagation(self, inputs, training):
        self.input = inputs
        self.output = np.dot(self.input, self.weights) + self.biases
        return self.output
 
    def backward_propagation(self, output_error):
         input_error = np.dot(output_error, self.weights.T)
         weights_error = np.dot(self.input.T, output_error)
         weights_error += self.l2_lambda * self.weights
         bias_error = np.sum(output_error, axis=0, keepdims=True)
         self.weights = self.w_opt.update(self.weights, weights_error)
         self.biases = self.b_opt.update(self.biases, bias_error)
         return input_error
 
    def output_shape(self):
         return (self.n_units,) 

In [5]:
class ActivationLayer(Layer):

    def forward_propagation(self, input, training):
        self.input = input
        self.output = self.activation_function(self.input)
        return self.output

    def backward_propagation(self, output_error):
        return self.derivative(self.input) * output_error

    @abstractmethod
    def activation_function(self, input):
        raise NotImplementedError

    @abstractmethod
    def derivative(self, input):
        raise NotImplementedError

    def output_shape(self):
        return self._input_shape

    def parameters(self):
        return 0

In [6]:
class ReLUActivation(ActivationLayer):

    def activation_function(self, input):
        return np.maximum(0, input)

    def derivative(self, input):
        return np.where(input >= 0, 1, 0)

A função Softmax, ao contrário da Sigmoid e da ReLU,  é geralmente usada na última camada da rede neural para converter as saídas da rede em probabilidades.  Essas probabilidades somam 1 (100%), sendo ideal para problemas de classificação multiclasse, como neste caso com 5 classes (Human, Anthropic, Google, OpenAI e Meta). 

Para garantir estabilidade numérica, foi utilizado o np.max para evitar que o np.exp resultasse em valores infinitos caso a entrada fosse muito alta.
A derivada de Softmax é complexa, uma vez que envolve uma matriz Jacobiana.  No entanto, ao utilizar Categorical Cross-Entropy como função loss, logo o cálculo do erro gradiente se simplificou significativamente.

In [7]:
class SoftmaxActivation(ActivationLayer):

    def activation_function(self, input):
        exps = np.exp(input - np.max(input, axis=-1, keepdims=True))
        return exps / np.sum(exps, axis=-1, keepdims=True)

    def derivative(self, input):
        return 1

In [8]:
class LossFunction:

    @abstractmethod
    def loss(self, y_true, y_pred):
        raise NotImplementedError

    @abstractmethod
    def derivative(self, y_true, y_pred):
        raise NotImplementedError

Para problemas de classificação multiclasse, a função de loss Categorical Cross-Entropy é a escolha padrão, pois penaliza de forma mais rigorosa previsões confiantes que estejam erradas. Teoricamente, a derivada da Cross-Entropy e a da Softmax são complexas separadamente.  No entanto, quando calculamos o erro que volta da última camada (Softmax) usando esta Loss, a fórmula resultante é apenas a diferença entre a previsão e o valor real.

In [9]:
class CategoricalCrossEntropy(LossFunction):

    def loss(self, y_true, y_pred):
        p = np.clip(y_pred, 1e-15, 1.0)
        return -np.sum(y_true * np.log(p)) / y_true.shape[0]

    def derivative(self, y_true, y_pred):
        n_samples = y_true.shape[0]
        return (y_pred - y_true) / n_samples

In [10]:
def accuracy(y_true, y_pred):
 
    def correct_format(y):
        y = np.array(y)
        if y.ndim > 1 and y.shape[1] > 1:
            return np.argmax(y, axis=1)
        elif y.ndim > 1 and y.shape[1] == 1:
            return np.round(y).flatten()
        return np.round(y)

    y_true = correct_format(y_true)
    y_pred = correct_format(y_pred)
    
    return np.mean(y_true == y_pred)

In [11]:
class Optimizer:

    def __init__(self, learning_rate = 0.01,  momentum = 0.90):
        self.retained_gradient = None
        self.learning_rate = learning_rate
        self.momentum = momentum
 
    def update(self, w, grad_loss_w):
        if self.retained_gradient is None:
            self.retained_gradient = np.zeros_like(w)
        self.retained_gradient = self.momentum * self.retained_gradient + grad_loss_w
        return w - self.learning_rate * self.retained_gradient

O Early Stopping é uma técnica de controlo do fluxo de treino, que monitoriza se o modelo está a aprender ao longo dos epochs ou se entrou em overfitting. Para isso, o Early Stopping monitoriza um conjunto de dados de validação separados, para garantir que o modelo não está apenas a decorar o treino. O treino é interrompido automaticamente quando não se verificam melhorias no desempenho durante um determinado número de epochs.

In [12]:
class EarlyStopping:

    def __init__(self, patience=5, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = np.inf
        self.counter = 0

    def check(self, current_loss):
        if current_loss < self.best_loss - self.min_delta:
            self.best_loss = current_loss
            self.counter = 0
            return False
        else:
            self.counter += 1
            if self.counter >= self.patience:
                return True
        return False

In [13]:
class NeuralNetwork:
 
    def __init__(self, epochs = 100, batch_size = 128, optimizer = None,
                 learning_rate = 0.01, momentum = 0.90, verbose = False, 
                 loss = CategoricalCrossEntropy,
                 metric:callable = accuracy):
        self.epochs = epochs
        self.batch_size = batch_size
        self.optimizer = Optimizer(learning_rate=learning_rate, momentum= momentum)
        self.verbose = verbose
        self.loss = loss()
        self.metric = metric

        # attributes
        self.layers = []
        self.history = {}

    def add(self, layer):
        if self.layers:
            layer.set_input_shape(input_shape=self.layers[-1].output_shape())
        if hasattr(layer, 'initialize'):
            layer.initialize(self.optimizer)
        self.layers.append(layer)
        return self

    def get_mini_batches(self, X, y = None,shuffle = True):
        n_samples = X.shape[0]
        indices = np.arange(n_samples)
        assert self.batch_size <= n_samples, "Batch size cannot be greater than the number of samples"
        if shuffle:
            np.random.shuffle(indices)
        for start in range(0, n_samples, self.batch_size):
            end = min(start + self.batch_size, n_samples)
            if y is not None:
                yield X[indices[start:end]], y[indices[start:end]]
            else:
                yield X[indices[start:end]], None

    def forward_propagation(self, X, training):
        output = X
        for layer in self.layers:
            output = layer.forward_propagation(output, training)
        return output

    def backward_propagation(self, output_error):
        error = output_error
        for layer in reversed(self.layers):
            error = layer.backward_propagation(error)
        return error

    def fit(self, dataset, val_dataset=None ,patience=None):
        X = dataset.X
        y = dataset.y
        if np.ndim(y) == 1:
            y = np.expand_dims(y, axis=1)

        self.history = {}

        if patience is not None and val_dataset is not None:
            early_stopping = EarlyStopping(patience=patience)

        for epoch in range(1, self.epochs + 1):
            output_x_ = []
            y_ = []
            for X_batch, y_batch in self.get_mini_batches(X, y):
                output = self.forward_propagation(X_batch, training=True)
                error = self.loss.derivative(y_batch, output)
                self.backward_propagation(error)

                output_x_.append(output)
                y_.append(y_batch)

            output_x_all = np.concatenate(output_x_)
            y_all = np.concatenate(y_)

            loss = self.loss.loss(y_all, output_x_all)
            val_loss_str = ""

            if self.metric is not None:
                metric = self.metric(y_all, output_x_all)
                metric_s = f"{self.metric.__name__}: {metric:.4f}"
            else:
                metric_s = "NA"
                metric = 'NA'

            if val_dataset is not None:
                val_predictions = self.predict(val_dataset)
                
                val_loss = self.loss.loss(val_dataset.y, val_predictions) 
                val_loss_str = f" - val_loss: {val_loss:.4f}"
                
                if patience is not None:
                    if early_stopping.check(val_loss):
                        if self.verbose: 
                            print(f"Epoch {epoch}/{self.epochs} - loss: {loss:.4f}{val_loss_str}")
                            print(f"Early stopping at epoch {epoch} with loss: {loss:.4f}")
                        break

            self.history[epoch] = {'loss': loss, 'metric': metric}

            if self.verbose:
                print(f"Epoch {epoch}/{self.epochs} - loss: {loss:.4f}{val_loss_str}")

        return self

    def predict(self, dataset):
        X = dataset.X if hasattr(dataset, 'X') else dataset
        return self.forward_propagation(X, training=False)

    def score(self, dataset):
        if self.metric is not None:
            predictions = self.predict(dataset)
            return self.metric(dataset.y, predictions)
        else:
            raise ValueError("No metric specified for the neural network.")

## Leitura e tratamento dos textos do dataset 

In [14]:
train = pd.read_csv('custom_dataset_v1_mini.csv', sep = ';')

In [15]:
train_texts = train['Text'].values.tolist()
train_labels = train['Label'].values.tolist()

Todos os textos que serão usados em todas as submissões terão entre 80 e 120 palavras.

In [16]:
def filter_by_length(texts, labels, min_words=80, max_words=120):
    filtered_texts = []
    filtered_labels = []
    
    for text, label in zip(texts, labels):
        word_count = len(text.split())
        
        if min_words <= word_count <= max_words:
            filtered_texts.append(text)
            filtered_labels.append(label)
            
    return filtered_texts, filtered_labels

In [17]:
train_texts_filtered, train_labels_filtered = filter_by_length(train_texts, train_labels)

Para treinar a rede com a função loss CategoricalCrossEntropy, foi necessário converter as labels de texto num formato que o NumPy conseguisse processar matematicamente.  Foi utilizado código adaptado do ficheiro examples-text-processing da aula (4), que já inclui uma base para a função one_hot_encode.  No entanto, essa implementação está voltada para tokens (palavras).  Assim, para as labels do dataset, foi desenvolvida uma função específica que mapeia cada uma das 5 classes para um vetor binário.

In [18]:
def one_hot_encode(labels_list):
    classes = ["Human", "Anthropic", "Google", "OpenAI", "Meta"]
    
    label_to_int = {label: i for i, label in enumerate(classes)}
    
    one_hot = np.zeros((len(labels_list), len(classes)))
    
    for i, label in enumerate(labels_list):
        if label in label_to_int:
            index = label_to_int[label]
            one_hot[i, index] = 1
        else:
            raise ValueError(f"The label: {label} was not found")
            
    return one_hot, classes

In [19]:
y_one_hot, labels = one_hot_encode(train_labels_filtered)

In [20]:
def tokenize(text):
    text = text.lower()
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-zA-Z]", " ", text)
    return text.split()

In [21]:
def build_vocab(texts, max_words):
    counter = Counter()

    for text in texts:
        counter.update(tokenize(text))

    most_common = counter.most_common(max_words)

    vocab = {"<pad>": 0, "<unk>": 1}
    for i, (word, _) in enumerate(most_common, start=2):
        vocab[word] = i

    return vocab

In [22]:
vocab = build_vocab(train_texts_filtered, max_words = 10000)

Para criar a lógica de Bag of Words (BoW), utilizamos as funções de processamento de texto que já existem no ficheiro examples-text-processing da aula (4)  e estendemo-las para gerar uma matriz numérica onde cada coluna representa a frequência de uma palavra do vocabulário.

In [23]:
def texts_to_bow(texts, vocab):
    vocab_size = len(vocab)
    n_texts = len(texts)
    
    bow_matrix = np.zeros((n_texts, vocab_size))
    
    for i, text in enumerate(texts):
        tokens = tokenize(text)
        
        for token in tokens:
            if token in vocab:
                index = vocab[token]
                bow_matrix[i, index] += 1
            else:
                bow_matrix[i, vocab["<unk>"]] += 1
                
    return bow_matrix

In [24]:
X_train = texts_to_bow(train_texts_filtered, vocab)

In [25]:
row_sums = X_train.sum(axis = 1, keepdims = True)
X_train_norm = (X_train / row_sums) * 100

In [26]:
dataset = Data(X = X_train_norm, y = y_one_hot)

In [27]:
def train_test_split(dataset, test_size=0.2, random_state=None):
    if random_state is not None:
        np.random.seed(random_state)

    indices = np.arange(len(dataset.X))
    np.random.shuffle(indices)
    split_idx = int(len(dataset.X) * (1 - test_size))

    train_indices = indices[:split_idx]
    test_indices = indices[split_idx:]

    train_data = Data(X=dataset.X[train_indices], y=dataset.y[train_indices] if dataset.has_label() else None, features=dataset.features, label=dataset.label)
    test_data = Data(X=dataset.X[test_indices], y=dataset.y[test_indices] if dataset.has_label() else None, features=dataset.features, label=dataset.label)

    return train_data, test_data

In [28]:
train_dataset, val_data = train_test_split(dataset, test_size=0.2, random_state=42)

In [29]:
n_features = X_train.shape[1]

## Modelo DNN

In [30]:
net = NeuralNetwork(
    epochs = 1000,
    batch_size = 32,
    learning_rate = 0.005,
    momentum = 0.8,
    verbose = True,
    loss = CategoricalCrossEntropy,
    metric = accuracy
)

In [31]:
net.add(DenseLayer(128, (n_features, ), l2_lambda=0.01))
net.add(ReLUActivation())

net.add(DenseLayer(64))
net.add(ReLUActivation())

net.add(DenseLayer(5, l2_lambda=0.001))
net.add(SoftmaxActivation())

In [32]:
net.fit(train_dataset, val_dataset=val_data, patience = 10)

Epoch 1/1000 - loss: 1.0386 - val_loss: 0.9449
Epoch 2/1000 - loss: 0.8751 - val_loss: 0.9084
Epoch 3/1000 - loss: 0.7863 - val_loss: 0.8744
Epoch 4/1000 - loss: 0.7259 - val_loss: 0.8689
Epoch 5/1000 - loss: 0.6671 - val_loss: 0.8559
Epoch 6/1000 - loss: 0.6217 - val_loss: 0.9894
Epoch 7/1000 - loss: 0.5772 - val_loss: 0.8523
Epoch 8/1000 - loss: 0.5434 - val_loss: 0.8501
Epoch 9/1000 - loss: 0.5059 - val_loss: 0.9445
Epoch 10/1000 - loss: 0.4916 - val_loss: 0.8509
Epoch 11/1000 - loss: 0.4615 - val_loss: 0.8766
Epoch 12/1000 - loss: 0.4331 - val_loss: 0.8588
Epoch 13/1000 - loss: 0.4219 - val_loss: 0.8536
Epoch 14/1000 - loss: 0.3981 - val_loss: 0.9853
Epoch 15/1000 - loss: 0.3738 - val_loss: 0.8539
Epoch 16/1000 - loss: 0.3548 - val_loss: 1.0430
Epoch 17/1000 - loss: 0.3527 - val_loss: 0.9984
Epoch 18/1000 - loss: 0.3463 - val_loss: 1.0044
Early stopping at epoch 18 with loss: 0.3463


In [33]:
print(f"Accuracy Final: {net.score(val_data):.4f}")

Accuracy Final: 0.6532


## Modelo de baseline de Regressão Logística

Em termos de Redes Neuronais, a Regressão Logística é simplesmente uma rede com zero camadas ocultas. Ou seja, os inputs ligam-se diretamente à camada de saída.

In [34]:
baseline_model = NeuralNetwork(
    epochs = 100,
    batch_size = 32,
    learning_rate = 0.01,
    momentum = 0.8,
    verbose = True,
    loss = CategoricalCrossEntropy,
    metric = accuracy
)

In [35]:
baseline_model.add(DenseLayer(5, (n_features, )))
baseline_model.add(SoftmaxActivation())

In [36]:
baseline_model.fit(train_dataset, val_dataset=val_data, patience = 10)

Epoch 1/100 - loss: 1.0430 - val_loss: 0.9837
Epoch 2/100 - loss: 0.9259 - val_loss: 0.9617
Epoch 3/100 - loss: 0.8770 - val_loss: 0.9319
Epoch 4/100 - loss: 0.8512 - val_loss: 0.9279
Epoch 5/100 - loss: 0.8356 - val_loss: 0.9651
Epoch 6/100 - loss: 0.8242 - val_loss: 0.9352
Epoch 7/100 - loss: 0.8186 - val_loss: 0.9420
Epoch 8/100 - loss: 0.8114 - val_loss: 0.9134
Epoch 9/100 - loss: 0.8136 - val_loss: 0.9346
Epoch 10/100 - loss: 0.8073 - val_loss: 0.9500
Epoch 11/100 - loss: 0.8034 - val_loss: 0.9329
Epoch 12/100 - loss: 0.8005 - val_loss: 0.9306
Epoch 13/100 - loss: 0.8066 - val_loss: 0.9233
Epoch 14/100 - loss: 0.7999 - val_loss: 0.9585
Epoch 15/100 - loss: 0.8004 - val_loss: 0.9761
Epoch 16/100 - loss: 0.8036 - val_loss: 0.9362
Epoch 17/100 - loss: 0.8032 - val_loss: 0.9232
Epoch 18/100 - loss: 0.7980 - val_loss: 0.9921
Early stopping at epoch 18 with loss: 0.7980


In [37]:
print(f"Accuracy Final: {baseline_model.score(val_data):.4f}")

Accuracy Final: 0.6191


## Teste dos modelos com o dataset-exemplos.csv

In [38]:
# Carregar o dataset de teste real (o que tem as labels para conferir)
df_exemplo = pd.read_csv('dataset-exemplos.csv', sep=';')

exemplo_texts = df_exemplo['Text'].values.tolist()
exemplo_labels = df_exemplo['Label'].values.tolist()

# 1. Transformar as labels em One-Hot (usando a mesma função e lista de classes do treino)
y_exemplo_one_hot, _ = one_hot_encode(exemplo_labels)

# 2. Transformar os textos em BoW usando o VOCABULÁRIO DO TREINO (Crucial!)
X_exemplo = texts_to_bow(exemplo_texts, vocab)

# 3. Aplicar a mesma normalização que usaste no treino
row_sums_ex = X_exemplo.sum(axis=1, keepdims=True)
X_exemplo_norm = (X_exemplo / row_sums_ex) * 100

# 4. Criar o objeto Data para facilitar o uso na rede
exemplo_dataset = Data(X=X_exemplo_norm, y=y_exemplo_one_hot)

In [39]:
acc_exemplo = net.score(exemplo_dataset)

print(fAccuracy dataset-exemplos: {acc_exemplo:.4f}")

preds_ex = net.predict(exemplo_dataset)
for i in range(5):
    p = np.argmax(preds_ex[i])
    r = np.argmax(y_exemplo_one_hot[i])
    print(f"Texto {i}: Previu {labels[p]} | Real: {labels[r]}")

SyntaxError: invalid decimal literal (2465773721.py, line 3)

In [ ]:
acc_exemplo = baseline_model.score(exemplo_dataset)

print(f"Accuracy dataset-exemplos: {acc_exemplo:.4f}")

preds_ex = baseline_model.predict(exemplo_dataset)
for i in range(5):
    p = np.argmax(preds_ex[i])
    r = np.argmax(y_exemplo_one_hot[i])
    print(f"Texto {i}: Previu {labels[p]} | Real: {labels[r]}")

## Submissão

In [ ]:
test = pd.read_csv('subm1.csv', sep = ';')

In [ ]:
X_test = texts_to_bow(test['Text'].values.tolist(), vocab)

In [ ]:
row_sums_test = X_test.sum(axis=1, keepdims=True)
X_test_norm = (X_test / row_sums_test) * 100

In [ ]:
predictions = net.predict(X_test_norm)

In [ ]:
predicted_indices = np.argmax(predictions, axis=1)
predicted_labels = [labels[i] for i in predicted_indices]

In [ ]:
submission = pd.DataFrame({
    'ID': test['ID'],
    'Text': test['Text'],
    'Label': predicted_labels
})

In [ ]:
submission.to_csv('subm1-g1-MMC-A.csv', sep=';', index = False)